Import libraries

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

def add_sys_path(path: Path | str) -> None:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)


root_libdir = subprocess.check_output(["root-config", "--libdir"], text=True).strip()
add_sys_path(root_libdir)

roounfold_root = Path(os.environ.get("ROOUNFOLD_ROOT", "/Users/gnigmat/work/RooUnfold"))
roounfold_src = roounfold_root / "src"
roounfold_build = roounfold_root / "build"
roounfold_lib = roounfold_build / "libRooUnfold.dylib"

for path in (roounfold_src, roounfold_build):
    add_sys_path(path)

if not roounfold_lib.exists():
    raise FileNotFoundError(f"RooUnfold library not found: {roounfold_lib}")

import ROOT

method = "bayes"
if len(sys.argv) > 1:
    method = sys.argv[1]

from ROOT import TH1D, TH2D, TCanvas, THnSparse, gSystem

ROOT.gSystem.Load(str(roounfold_lib))

In [ ]:
# Set the ROOT style
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetOptTitle(0)
ROOT.gStyle.SetPalette(ROOT.kBird)

In [ ]:
# Define useful functions
text = ROOT.TLatex()
text.SetTextFont(42)
text.SetTextSize(0.04)

# Plot CMS header on the canvas
def plotCMSHeader(collSystem=0, energy=8.16):
    # collSystem: 0 = pp, 1 = pPb, 2 = PbPb
    # energy in TeV
    collSystemStr = "pp" if collSystem == 0 else "pPb" if collSystem == 1 else "PbPb"
    t = ROOT.TLatex()
    t.SetTextFont(42)
    t.SetTextSize(0.05)
    t.DrawLatexNDC(0.15, 0.93, "#bf{CMS} #it{Preliminary}")
    t.SetTextSize(0.04)
    t.DrawLatexNDC(0.6, 0.93, f"{collSystemStr} #sqrt{{s_{{NN}}}} = {energy:.2f} TeV")
    t.SetTextSize(0.05)


# Set the pad style
def setPadStyle():
    ROOT.gPad.SetTopMargin(0.1)
    ROOT.gPad.SetBottomMargin(0.15)
    ROOT.gPad.SetRightMargin(0.12)
    ROOT.gPad.SetLeftMargin(0.15)


# Set the style for 1D histograms
def set1DStyle(h, type=0, doRenorm=False):
    markerStyle = 20
    markerSize = 1.3
    lineWidth = 2
    color = 2
    markerStyles = [20, 21, 20, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]
    p8Colors = ["kRed", "kBlue", "kBlack", "kMagenta", "kP8Orange", "kP8Green", "kP8Azure", "kPink", "kCyan", "kTeal", "kGray", "kSpring", "kViolet"]

    if type < len(markerStyles):
        color_name = p8Colors[type]
        color_value = getattr(ROOT, color_name)
        h.SetLineColor(color_value)
        h.SetMarkerColor(color_value)
        h.SetMarkerStyle(markerStyles[type])
    else:
        color = 6
        markerStyle = 45
        h.SetMarkerColor(color)
        h.SetLineColor(color)
        h.SetMarkerStyle(markerStyle)

    h.SetLineWidth(lineWidth)
    h.SetMarkerSize(markerSize)

    h.GetYaxis().SetTitleSize(0.05)
    h.GetYaxis().SetLabelSize(0.05)
    h.GetXaxis().SetTitleSize(0.05)
    h.GetXaxis().SetLabelSize(0.05)
    h.GetXaxis().SetTitleOffset(1.2)
    h.GetXaxis().SetNdivisions(208)
    h.GetYaxis().SetTitleOffset(1.2)
    h.GetYaxis().SetNdivisions(208)

    if doRenorm:
        integral = h.Integral()
        if integral > 0:
            h.Scale(1.0 / integral)

# Set the style for 2D histograms
def set2DStyle(h):
    h.GetYaxis().SetTitleSize(0.05)
    h.GetYaxis().SetLabelSize(0.05)
    h.GetXaxis().SetTitleSize(0.05)
    h.GetXaxis().SetLabelSize(0.05)
    h.GetXaxis().SetTitleOffset(1.2)
    h.GetXaxis().SetNdivisions(208)
    h.GetYaxis().SetTitleOffset(1.3)
    h.GetYaxis().SetNdivisions(208)


Import RooUnfold

In [ ]:
try:
    import RooUnfold
except ImportError as exc:
    raise ImportError(
        f"Unable to import RooUnfold after loading {roounfold_lib}. "
        f"Check that ROOUNFOLD_ROOT points to the RooUnfold checkout."
    ) from exc

In [ ]:
generator_name = "embedding"
generator_label = generator_name.capitalize()
fname = f"/Users/gnigmat/cernbox/ana/pPb8160/{generator_name}/{generator_name}_jetId.root"
eta_cuts = [1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.5]

generator_label

In [ ]:
input_file = ROOT.TFile.Open(fname)
if not input_file or input_file.IsZombie():
    raise FileNotFoundError(f"Unable to open ROOT file: {fname}")

n_eta_cuts = len(eta_cuts)

genPtEtaCM = []
recoPtEtaCM = []
genPtEtaCMMiss = []
genPtEtaCMVsRecoPtEtaCM = []

for i in range(n_eta_cuts):
    gen_hist = input_file.Get(f"hGenDijetPtEtaCM_{i}")
    reco_hist = input_file.Get(f"hRecoDijetPtEtaCM_{i}")
    miss_hist = input_file.Get(f"hGenDijetPtEtaCMMiss_{i}")
    response_hist = input_file.Get(f"hGenDijetPtEtaCMVsRecoPtEtaCM_{i}").Clone(f"hGenDijetPtEtaCMVsRecoPtEtaCM_{i}")


    if gen_hist is None:
        raise KeyError(f"Missing histogram hGenDijetPtEtaCM_{i} in {fname}")
    if reco_hist is None:
        raise KeyError(f"Missing histogram hRecoDijetPtEtaCM_{i} in {fname}")
    if miss_hist is None:
        raise KeyError(f"Missing histogram hGenDijetPtEtaCMMiss_{i} in {fname}")
    if response_hist is None:
        raise KeyError(f"Missing histogram hGenDijetPtEtaCMVsRecoPtEtaCM_{i} in {fname}")
    
    gen_hist.SetDirectory(0)
    reco_hist.SetDirectory(0)
    miss_hist.SetDirectory(0)
    # response_hist.SetDirectory(0)

    genPtEtaCM.append(gen_hist)
    recoPtEtaCM.append(reco_hist)
    genPtEtaCMMiss.append(miss_hist)
    genPtEtaCMVsRecoPtEtaCM.append(response_hist)

input_file.Close()

In [ ]:
# # For test purpose only, plot the first histogram

# if not genPtEtaCM:
#     raise RuntimeError("genPtEtaCM is empty; load the ROOT file first")

# canvas_name = "canvas_gen_pt_eta_cm_0"
# existing_canvas = ROOT.gROOT.FindObject(canvas_name)
# if existing_canvas:
#     existing_canvas.Close()

# canvas_gen_dijet_pt_eta_cm_0 = ROOT.TCanvas(canvas_name, "cGenDijetPtEtaCM_0", 800, 800)
# canvas_gen_dijet_pt_eta_cm_0.cd()
# setPadStyle()
# set2DStyle(genPtEtaCM[0])
# genPtEtaCM[0].Draw("COLZ")
# canvas_gen_dijet_pt_eta_cm_0.Modified()
# canvas_gen_dijet_pt_eta_cm_0.Update()
# canvas_gen_dijet_pt_eta_cm_0

In [ ]:
pt_ave_bins = [60, 80, 180, 250, 300, 500]

# 1D projections of the 2D histograms for each pt_ave bin
genEtaCM = []
recoEtaCM = []
genEtaCMMiss = []

# 2D projections of the THnSparse histograms for each pt_ave bin
gen2recoResponse = []

for i in range(len(eta_cuts)):
    gen_proj = []
    reco_proj = []
    miss_proj = []
    gen2reco_proj = []

    for j, (low_val, high_val) in enumerate(zip(pt_ave_bins[:-1], pt_ave_bins[1:])):
        low_bin = int(genPtEtaCM[0].GetXaxis().FindBin(low_val + 0.01))
        high_bin = int(genPtEtaCM[0].GetXaxis().FindBin(high_val - 0.01))


        # Project the 2D histograms onto the Y-axis (eta) for the given pt_ave bin
        gen_hist = genPtEtaCM[i].ProjectionY(f"hGenDijetEtaCM_{i}_{j}", low_bin, high_bin)
        reco_hist = recoPtEtaCM[i].ProjectionY(f"hRecoDijetEtaCM_{i}_{j}", low_bin, high_bin)
        miss_hist = genPtEtaCMMiss[i].ProjectionY(f"hGenDijetEtaCMMiss_{i}_{j}", low_bin, high_bin)
        # Set the range for the THnSparse histogram to project onto the Y-axis (eta) for the given pt_ave bin
        genPtEtaCMVsRecoPtEtaCM[i].GetAxis(0).SetRange(low_bin, high_bin)
        genPtEtaCMVsRecoPtEtaCM[i].GetAxis(2).SetRange(low_bin, high_bin)
        gen2reco_hist = genPtEtaCMVsRecoPtEtaCM[i].Projection(3, 1)
        gen2reco_hist.SetName(f"hGen2RecoDijetEtaCM_{i}_{j}")

        gen_hist.SetDirectory(0)
        reco_hist.SetDirectory(0)
        miss_hist.SetDirectory(0)
        gen2reco_hist.SetDirectory(0)

        gen_proj.append(gen_hist)
        reco_proj.append(reco_hist)
        miss_proj.append(miss_hist)
        gen2reco_proj.append(gen2reco_hist)

    genEtaCM.append(gen_proj)
    recoEtaCM.append(reco_proj)
    genEtaCMMiss.append(miss_proj)
    gen2recoResponse.append(gen2reco_proj)


In [ ]:
# # Plot all genEtaCM projections (all pt_ave bins) on one canvas for a given eta cut
# if not genEtaCM:
#     raise RuntimeError("genEtaCM is empty; run the projection cell first")

# eta_idx = 5  # choose eta-cut index here
# if eta_idx < 0 or eta_idx >= len(eta_cuts):
#     raise IndexError(f"eta_idx={eta_idx} is out of range for eta_cuts")

# canvas_name = f"canvas_genEtaCM_allPt_eta{eta_idx}"
# existing_canvas = ROOT.gROOT.FindObject(canvas_name)
# if existing_canvas:
#     existing_canvas.Close()

# canvas_genEtaCM_allPt = ROOT.TCanvas(canvas_name, f"Gen etaCM projections (eta idx {eta_idx})", 800, 800)
# canvas_genEtaCM_allPt.cd()
# setPadStyle()

# gen_eta_overlays = []
# for pt_idx in range(len(pt_ave_bins) - 1):
#     hist = genEtaCM[eta_idx][pt_idx].Clone(f"hGenEtaCM_overlay_eta{eta_idx}_pt{pt_idx}")
#     hist.SetDirectory(0)
#     set1DStyle(hist, pt_idx, True)
#     hist.SetTitle(";#eta_{CM};Normalized entries")
#     draw_opt = "E" if pt_idx == 0 else "E SAME"
#     hist.Draw(draw_opt)
#     gen_eta_overlays.append(hist)

# legend = ROOT.TLegend(0.6, 0.75, 0.88, 0.88)
# legend.SetBorderSize(0)
# legend.SetFillStyle(0)
# legend.SetTextFont(42)
# legend.SetTextSize(0.03)
# for pt_idx in range(len(pt_ave_bins) - 1):
#     label = f"{pt_ave_bins[pt_idx]} < p_{{T}}^{{ave}} < {pt_ave_bins[pt_idx + 1]} GeV"
#     legend.AddEntry(gen_eta_overlays[pt_idx], label, "p")
# legend.Draw()

# text = ROOT.TLatex()
# text.SetNDC(True)
# text.SetTextFont(42)
# text.SetTextSize(0.04)
# text.DrawLatex(0.16, 0.92, f"Gen #eta_{{CM}} projections, eta-cut index = {eta_idx}")

# canvas_genEtaCM_allPt.Modified()
# canvas_genEtaCM_allPt.Update()
# canvas_genEtaCM_allPt

In [ ]:
# Plot the first 1D projection and response for test purpose
if not genEtaCM:
    raise RuntimeError("genEtaCM projections are empty; run the projection cell first")
if not recoEtaCM:
    raise RuntimeError("recoEtaCM projections are empty; run the projection cell first")
if not genEtaCMMiss:
    raise RuntimeError("genEtaCMMiss projections are empty; run the projection cell first")
if not gen2recoResponse:
    raise RuntimeError("gen2recoResponse is empty; run the projection cell first")

eta_idx = 5
pt_bin_idx = 0

h_gen_eta = genEtaCM[eta_idx][pt_bin_idx]
h_reco_eta = recoEtaCM[eta_idx][pt_bin_idx]
h_miss_eta = genEtaCMMiss[eta_idx][pt_bin_idx]
h_response = gen2recoResponse[eta_idx][pt_bin_idx]

canvas_name = "canvas_test_proj_resp"
existing_canvas = ROOT.gROOT.FindObject(canvas_name)
if existing_canvas:
    existing_canvas.Close()

canvas_test = ROOT.TCanvas(canvas_name, "Projection and Response", 1400, 600)
canvas_test.Divide(2, 1)

canvas_test.cd(1)
setPadStyle()
set1DStyle(h_gen_eta, 0, False)
set1DStyle(h_reco_eta, 1, False)
set1DStyle(h_miss_eta, 2, False)

h_gen_eta.SetTitle(";#eta_{CM};Entries")
h_gen_eta.Draw("E")
h_reco_eta.Draw("E SAME")
h_miss_eta.Draw("E SAME")

h_gen_eta.GetXaxis().SetRangeUser( -eta_cuts[eta_idx] - 0.1, eta_cuts[eta_idx] + 0.1)

text.DrawLatexNDC(0.18, 0.85, f"{generator_label}")
text.SetTextSize(0.03)
text.DrawLatexNDC(0.18, 0.8, f"{pt_ave_bins[pt_bin_idx]} < p_{{T}}^{{ave}} < {pt_ave_bins[pt_bin_idx + 1]} GeV")
text.DrawLatexNDC(0.18, 0.75, f"p_{{T}}^{{Lead}} > 50 GeV")
text.DrawLatexNDC(0.18, 0.70, f"p_{{T}}^{{SubLead}} > 50 GeV")
text.DrawLatexNDC(0.18, 0.65, f"|#eta_{{CM}}| < {eta_cuts[eta_idx]}")
text.DrawLatexNDC(0.18, 0.60, f"|#Delta#phi| > 2#pi/3")
text.SetTextSize(0.04)

legend = ROOT.TLegend(0.6, 0.68, 0.88, 0.88)
legend.SetBorderSize(0)
legend.SetFillStyle(0)
legend.AddEntry(h_gen_eta, "Gen", "p")
legend.AddEntry(h_reco_eta, "Reco", "p")
legend.AddEntry(h_miss_eta, "Gen miss", "p")
legend.Draw()

canvas_test.cd(2)
setPadStyle()
set2DStyle(h_response)
h_response.SetTitle(";Reco #eta_{CM};Gen #eta_{CM}")
h_response.Draw("COLZ")
h_response.GetXaxis().SetRangeUser( -eta_cuts[eta_idx] - 0.1, eta_cuts[eta_idx] + 0.1)
h_response.GetYaxis().SetRangeUser( -eta_cuts[eta_idx] - 0.1, eta_cuts[eta_idx] + 0.1)

text.DrawLatexNDC(0.18, 0.85, f"{generator_label}")
text.SetTextSize(0.03)
text.DrawLatexNDC(0.18, 0.8, f"{pt_ave_bins[pt_bin_idx]} < p_{{T}}^{{ave}} < {pt_ave_bins[pt_bin_idx + 1]} GeV")
text.DrawLatexNDC(0.18, 0.75, f"p_{{T}}^{{Lead}} > 50 GeV")
text.DrawLatexNDC(0.18, 0.70, f"p_{{T}}^{{SubLead}} > 50 GeV")
text.DrawLatexNDC(0.18, 0.65, f"|#eta_{{CM}}| < {eta_cuts[eta_idx]}")
text.DrawLatexNDC(0.18, 0.60, f"|#Delta#phi| > 2#pi/3")
text.SetTextSize(0.04)

canvas_test.Modified()
canvas_test.Update()
canvas_test

In [ ]:
# Build flattened etaCM histograms and response matrix for one eta-cut
if not genEtaCM or not recoEtaCM or not genEtaCMMiss or not gen2recoResponse:
    raise RuntimeError("Projection containers are empty; run the projection cell first")

eta_idx = 5  # choose eta-cut index here
if eta_idx < 0 or eta_idx >= len(eta_cuts):
    raise IndexError(f"eta_idx={eta_idx} is out of range for eta_cuts")

nPtSelections = len(pt_ave_bins) - 1
nEtaBins = genEtaCM[eta_idx][0].GetNbinsX()
nGlobalBins = nPtSelections * nEtaBins

hGenTruthEtaCM = ROOT.TH1D("hGenTruthEtaCM", ";global #eta_{CM} bin;Entries", nGlobalBins, 0.5, nGlobalBins + 0.5)
hGenTrusthEtaCMMiss = ROOT.TH1D("hGenTrusthEtaCMMiss", ";global #eta_{CM} bin;Entries", nGlobalBins, 0.5, nGlobalBins + 0.5)
hRecoMeasuredEtaCM = ROOT.TH1D("hRecoMeasuredEtaCM", ";global #eta_{CM} bin;Entries", nGlobalBins, 0.5, nGlobalBins + 0.5)

hResponseEtaCM = ROOT.TH2D(
    "hResponseEtaCM",
    ";global reco #eta_{CM} bin;global gen #eta_{CM} bin",
    nGlobalBins,
    0.5,
    nGlobalBins + 0.5,
    nGlobalBins,
    0.5,
    nGlobalBins + 0.5,
)

for pt_idx in range(nPtSelections):
    h_gen = genEtaCM[eta_idx][pt_idx]
    h_reco = recoEtaCM[eta_idx][pt_idx]
    h_miss = genEtaCMMiss[eta_idx][pt_idx]

    if h_gen.GetNbinsX() != nEtaBins or h_reco.GetNbinsX() != nEtaBins or h_miss.GetNbinsX() != nEtaBins:
        raise ValueError("Inconsistent number of eta bins across 1D projections")

    # Map local eta bins to global bins: [pt block][eta bin]
    for eta_bin in range(1, nEtaBins + 1):
        global_bin = pt_idx * nEtaBins + eta_bin

        hGenTruthEtaCM.SetBinContent(global_bin, h_gen.GetBinContent(eta_bin))
        hGenTruthEtaCM.SetBinError(global_bin, h_gen.GetBinError(eta_bin))

        hRecoMeasuredEtaCM.SetBinContent(global_bin, h_reco.GetBinContent(eta_bin))
        hRecoMeasuredEtaCM.SetBinError(global_bin, h_reco.GetBinError(eta_bin))

        hGenTrusthEtaCMMiss.SetBinContent(global_bin, h_miss.GetBinContent(eta_bin))
        hGenTrusthEtaCMMiss.SetBinError(global_bin, h_miss.GetBinError(eta_bin))

# Fill flattened response matrix including pt-migration off-diagonal blocks
sparse_resp = genPtEtaCMVsRecoPtEtaCM[eta_idx]
for gen_pt_idx, (gen_low, gen_high) in enumerate(zip(pt_ave_bins[:-1], pt_ave_bins[1:])):
    gen_low_bin = int(genPtEtaCM[0].GetXaxis().FindBin(gen_low + 0.01))
    gen_high_bin = int(genPtEtaCM[0].GetXaxis().FindBin(gen_high - 0.01))

    sparse_resp.GetAxis(0).SetRange(gen_low_bin, gen_high_bin)

    for reco_pt_idx, (reco_low, reco_high) in enumerate(zip(pt_ave_bins[:-1], pt_ave_bins[1:])):
        reco_low_bin = int(recoPtEtaCM[0].GetXaxis().FindBin(reco_low + 0.01))
        reco_high_bin = int(recoPtEtaCM[0].GetXaxis().FindBin(reco_high - 0.01))

        sparse_resp.GetAxis(2).SetRange(reco_low_bin, reco_high_bin)
        h_resp_block = sparse_resp.Projection(3, 1)
        h_resp_block.SetName(f"hResponseBlock_genPt{gen_pt_idx}_recoPt{reco_pt_idx}")

        for gen_eta_bin in range(1, nEtaBins + 1):
            global_gen_bin = gen_pt_idx * nEtaBins + gen_eta_bin
            for reco_eta_bin in range(1, nEtaBins + 1):
                global_reco_bin = reco_pt_idx * nEtaBins + reco_eta_bin

                content = h_resp_block.GetBinContent(reco_eta_bin, gen_eta_bin)
                error = h_resp_block.GetBinError(reco_eta_bin, gen_eta_bin)

                hResponseEtaCM.SetBinContent(global_reco_bin, global_gen_bin, content)
                hResponseEtaCM.SetBinError(global_reco_bin, global_gen_bin, error)

# Reset sparse axis ranges to full range
sparse_resp.GetAxis(0).SetRange(0, -1)
sparse_resp.GetAxis(2).SetRange(0, -1)

hGenTruthEtaCM.SetDirectory(0)
hGenTrusthEtaCMMiss.SetDirectory(0)
hRecoMeasuredEtaCM.SetDirectory(0)
hResponseEtaCM.SetDirectory(0)

print(f"Built flattened histograms for eta_idx={eta_idx}")
print(f"nPtSelections={nPtSelections}, nEtaBins={nEtaBins}, nGlobalBins={nGlobalBins}")
print("Response matrix now includes pt-migration off-diagonal blocks")

In [ ]:
# Plot 1D flattened histograms and 2D response matrix for test purpose

canvas_name = "canvas_flattened"
existing_canvas = ROOT.gROOT.FindObject(canvas_name)
if existing_canvas:
    existing_canvas.Close()

canvas_flattened = ROOT.TCanvas("canvas_flattened", "Flattened Histograms and Response", 1400, 600)
canvas_flattened.Divide(2, 1)

canvas_flattened.cd(1)
setPadStyle()
set1DStyle(hGenTruthEtaCM, 0, False)
set1DStyle(hRecoMeasuredEtaCM, 1, False)
set1DStyle(hGenTrusthEtaCMMiss, 2, False)
hGenTruthEtaCM.SetTitle(";#eta_{CM} bin;Entries")
hGenTruthEtaCM.Draw("E")
hRecoMeasuredEtaCM.Draw("E SAME")
hGenTrusthEtaCMMiss.Draw("E SAME")

legend_flat = ROOT.TLegend(0.6, 0.68, 0.88, 0.88)
legend_flat.SetBorderSize(0)
legend_flat.SetFillStyle(0)
legend_flat.AddEntry(hGenTruthEtaCM, "Gen", "p")
legend_flat.AddEntry(hRecoMeasuredEtaCM, "Reco", "p")
legend_flat.AddEntry(hGenTrusthEtaCMMiss, "Gen miss", "p")
legend_flat.Draw()

text.DrawLatexNDC(0.18, 0.85, f"{generator_label}")
text.DrawLatexNDC(0.45, 0.92, f"|#eta_{{CM}}| < {eta_cuts[eta_idx]}")

canvas_flattened.cd(2)
setPadStyle()
set2DStyle(hResponseEtaCM)
hResponseEtaCM.SetTitle(";Reco #eta_{CM} bin;Gen #eta_{CM} bin")
hResponseEtaCM.Draw("COLZ")
text.DrawLatexNDC(0.45, 0.92, f"|#eta_{{CM}}| < {eta_cuts[eta_idx]}")

canvas_flattened.Modified()
canvas_flattened.Update()
canvas_flattened

In [ ]:
# Prepare and run the unfolding using RooUnfold
if not hResponseEtaCM or not hRecoMeasuredEtaCM:
    raise RuntimeError("Response matrix or measured histogram is empty; run the previous cell first")
if not hGenTruthEtaCM:
    raise RuntimeError("Truth histogram is empty; run the previous cell first")
if not hGenTrusthEtaCMMiss:
    raise RuntimeError("Missed truth histogram is empty; run the previous cell first")
if not hRecoMeasuredEtaCM:
    raise RuntimeError("Measured histogram is empty; run the previous cell first")

# Create the RooUnfoldResponse object
response = ROOT.RooUnfoldResponse(hRecoMeasuredEtaCM, hGenTruthEtaCM, hResponseEtaCM)

# # Fill missed truth histogram into the response
# for bin_idx in range(1, hGenTrusthEtaCMMiss.GetNbinsX() + 1):
#     bin_center = hGenTrusthEtaCMMiss.GetBinCenter(bin_idx)
#     missed_content = hGenTrusthEtaCMMiss.GetBinContent(bin_idx)
#     # missed_error = hGenTrusthEtaCMMiss.GetBinError(bin_idx)

#     if missed_content <= 0:
#         continue  # Skip bins with zero or negative content
    
#     response.Miss(bin_center, missed_content)

# Set number of iterations for Bayesian unfolding
n_iterations = 4

unfold = ROOT.RooUnfoldBayes(response, hRecoMeasuredEtaCM, n_iterations)
hUnfoldedEtaCM = unfold.Hreco()
hUnfoldedEtaCM.SetName("hUnfoldedEtaCM")
hUnfoldedEtaCM.SetTitle(";#eta_{CM} bin;Entries")
hUnfoldedEtaCM.SetDirectory(0)

# Compute the covariance matrix of the unfolded result
# covariance_matrix = unfold.Ereco()

In [ ]:
# Plot the comparison of the unfolded result with the truth and measured histograms on one histogram
# and the ratio of unfolded and measured to truth on another histogram
canvas_name = "canvas_unfolded"
existing_canvas = ROOT.gROOT.FindObject(canvas_name)
if existing_canvas:
    existing_canvas.Close()

canvas_unfolded = ROOT.TCanvas(canvas_name, "Unfolded Result Comparison", 1400, 600)
canvas_unfolded.Divide(2, 1)

canvas_unfolded.cd(1)
setPadStyle()
set1DStyle(hGenTruthEtaCM, 0, False)
set1DStyle(hRecoMeasuredEtaCM, 1, False)
set1DStyle(hUnfoldedEtaCM, 2, False)
hGenTruthEtaCM.Draw("E")
hRecoMeasuredEtaCM.Draw("E SAME")
hUnfoldedEtaCM.Draw("E SAME")

legend_unfolded = ROOT.TLegend(0.6, 0.68, 0.88, 0.88)
legend_unfolded.SetBorderSize(0)
legend_unfolded.SetFillStyle(0)
legend_unfolded.AddEntry(hGenTruthEtaCM, "Gen", "p")
legend_unfolded.AddEntry(hRecoMeasuredEtaCM, "Reco", "p")
legend_unfolded.AddEntry(hUnfoldedEtaCM, f"Unfolded", "p")
legend_unfolded.Draw()

text.DrawLatexNDC(0.18, 0.85, f"{generator_label}")
text.DrawLatexNDC(0.45, 0.92, f"|#eta_{{CM}}| < {eta_cuts[eta_idx]}")

canvas_unfolded.cd(2)
setPadStyle()
hMeasuredToTruth = hRecoMeasuredEtaCM.Clone("hMeasuredToTruth")
hMeasuredToTruth.Divide(hGenTruthEtaCM)
hUnfoldedToTruth = hUnfoldedEtaCM.Clone("hUnfoldedToTruth")
hUnfoldedToTruth.Divide(hGenTruthEtaCM)

hMeasuredToTruth.SetTitle(";#eta_{CM} bin;Ratio to Gen")
hMeasuredToTruth.Draw("E")
hUnfoldedToTruth.Draw("E SAME")
hMeasuredToTruth.GetYaxis().SetRangeUser(0.5, 2.5)

legend_ratio = ROOT.TLegend(0.6, 0.68, 0.88, 0.88)
legend_ratio.SetBorderSize(0)
legend_ratio.SetFillStyle(0)
legend_ratio.AddEntry(hMeasuredToTruth, "Reco / Gen", "p")
legend_ratio.AddEntry(hUnfoldedToTruth, f"Unfolded / Gen", "p")
legend_ratio.Draw()

text.DrawLatexNDC(0.18, 0.85, f"{generator_label}")
text.DrawLatexNDC(0.45, 0.92, f"|#eta_{{CM}}| < {eta_cuts[eta_idx]}")

canvas_unfolded.Modified()
canvas_unfolded.Update()
canvas_unfolded